## Map Lexicon Chapters and Decoding Codes to IHRA Sections

- Run only once
- produces the file `lexicon_ihra.pkl`

In [1]:
import pandas as pd
from os.path import join

import sys
sys.path.append("..")

from config import PROJECT_DIR
from utils.data_helpers import convert_to_int
from utils.classification_helpers import split_ambiguous_sections


In [2]:
decoding_lexicon_ihra = pd.read_csv(join(PROJECT_DIR, "llm_prompting", "external_ressources", "decoding_ihra_lexicon_mapping.csv"))
decoding_lexicon_ihra.head(2)

,decoding_code_detailled,chapter_lexicon,ihra_section,decoding_short
0,S1,2,2,a12
1,S2,3,27,a12


In [3]:
decoding_lexicon_ihra.chapter_lexicon = decoding_lexicon_ihra.chapter_lexicon.map(convert_to_int)
decoding_lexicon_ihra.ihra_section = decoding_lexicon_ihra.ihra_section.map(convert_to_int)

In [4]:
# Only mapping of lexicon chapters to IHRA sections
lexicon_ihra = decoding_lexicon_ihra[["chapter_lexicon", "ihra_section"]].drop_duplicates()
lexicon_ihra

,chapter_lexicon,ihra_section
0,2,2
1,3,27
2,3,2
4,32,X
5,31,7
6,6,2
7,7,2
11,11,2
13,12,2
19,5,2


**Explanations:**

- IHRA Section 0: hatred against Jews (general paragraph before specific IHRA sections). Valid numbers are 0-11
- 'X' in column `chapter_lexicon` results from maping the decoding code `D16` because it was not included in the Lexicon. It is mentioned in various chapters ("victim-perpetrator reversal") but not a sub-chapter on its own
- 'X' in column `ihra_section` means there is no matching IHRA section
- Codes 27 and 29 in `ihra_section` result from ambiguity with regard to reference (Jews or Israel), and should be understood as "2 or 7" resp. "2 or 9".
- Multiple codes in `ihra_section` result from fine-grained codes from Decoding project mapping to different IHRA sections. Thus, all should be considered plausible.

We handle these as follows:
- Drop the row containing "X" in `chapter_lexicon` as this will not occur as a chapter reference
- replace "X" in `ihra_section` by NaN
- replace 27 by [2,7] and 29 by [2,9] in `ihra_section`
- Manually add 37, 38 and 42 as a chapter with `None` as IHRA section reference

In [5]:
lexicon_ihra = lexicon_ihra[lexicon_ihra["chapter_lexicon"]!="X"]
lexicon_ihra.sort_values(by="chapter_lexicon")

,chapter_lexicon,ihra_section
0,2,2
1,3,27
2,3,2
32,4,29
19,5,2
6,6,2
7,7,2
24,8,2
33,9,6
29,10,X


In [6]:
lexicon_ihra.replace({"ihra_section": {"X": None}}, inplace=True)

/var/folders/5r/147qn_r949n270dn6qvhgnrw0000gn/T/ipykernel_85226/2890047458.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lexicon_ihra.replace({"ihra_section": {"X": None}}, inplace=True)


In [7]:
lexicon_ihra

,chapter_lexicon,ihra_section
0,2,2
1,3,27
2,3,2
4,32,None
5,31,7
6,6,2
7,7,2
11,11,2
13,12,2
19,5,2


In [8]:
lexicon_ihra = pd.concat(
    [
        lexicon_ihra.sort_values(by="chapter_lexicon"), 
        pd.DataFrame([[37, None],[38, None],[42, None]], columns=["chapter_lexicon", "ihra_section"])
    ], axis=0).reset_index(drop=True)

In [9]:
lexicon_ihra

,chapter_lexicon,ihra_section
0,2,2
1,3,27
2,3,2
3,4,29
4,5,2
5,6,2
6,7,2
7,8,2
8,9,6
9,10,None


In [10]:
lexicon_ihra = lexicon_ihra.groupby("chapter_lexicon")["ihra_section"].apply(list).to_dict()
lexicon_ihra

{2: [2],
 3: [27, 2],
 4: [29],
 5: [2],
 6: [2],
 7: [2],
 8: [2],
 9: [6],
 10: [None],
 11: [2],
 12: [2],
 13: [2, 29],
 14: [2],
 15: [4],
 16: [4],
 17: [None],
 18: [4],
 19: [None],
 20: [None],
 21: [4],
 22: [None],
 23: [27],
 24: [10],
 25: [11],
 26: [27],
 27: [0, 1],
 28: [10],
 29: [2, 7],
 30: [7],
 31: [7],
 32: [None],
 33: [8],
 34: [7],
 35: [7],
 36: [9],
 37: [None],
 38: [None],
 39: [None],
 40: [0],
 41: [0],
 42: [None]}

In [12]:
lexicon_ihra = {k: split_ambiguous_sections(v) for k,v in lexicon_ihra.items()}
lexicon_ihra

{2: [2],
 3: [2, 7],
 4: [2, 9],
 5: [2],
 6: [2],
 7: [2],
 8: [2],
 9: [6],
 10: [None],
 11: [2],
 12: [2],
 13: [2, 9],
 14: [2],
 15: [4],
 16: [4],
 17: [None],
 18: [4],
 19: [None],
 20: [None],
 21: [4],
 22: [None],
 23: [2, 7],
 24: [10],
 25: [11],
 26: [2, 7],
 27: [0, 1],
 28: [10],
 29: [2, 7],
 30: [7],
 31: [7],
 32: [None],
 33: [8],
 34: [7],
 35: [7],
 36: [9],
 37: [None],
 38: [None],
 39: [None],
 40: [0],
 41: [0],
 42: [None]}

In [13]:
lexicon_ihra = {k:[item for item in v if pd.notnull(item)] for k,v in lexicon_ihra.items()}
lexicon_ihra = {k:v for k,v in lexicon_ihra.items() if len(v)>0}
lexicon_ihra = {k:[int(item) for item in v] for k,v in lexicon_ihra.items()}

In [14]:
lexicon_ihra

{2: [2],
 3: [2, 7],
 4: [2, 9],
 5: [2],
 6: [2],
 7: [2],
 8: [2],
 9: [6],
 11: [2],
 12: [2],
 13: [2, 9],
 14: [2],
 15: [4],
 16: [4],
 18: [4],
 21: [4],
 23: [2, 7],
 24: [10],
 25: [11],
 26: [2, 7],
 27: [0, 1],
 28: [10],
 29: [2, 7],
 30: [7],
 31: [7],
 33: [8],
 34: [7],
 35: [7],
 36: [9],
 40: [0],
 41: [0]}

In [15]:
ihra_lexicon_df = pd.DataFrame(lexicon_ihra.items())
ihra_lexicon_df.columns = ["ihra_section", "lexicon_chapter"]
ihra_lexicon_df

,ihra_section,lexicon_chapter
0,2,[2]
1,3,"[2, 7]"
2,4,"[2, 9]"
3,5,[2]
4,6,[2]
5,7,[2]
6,8,[2]
7,9,[6]
8,11,[2]
9,12,[2]


In [16]:
ihra_lexicon_df.to_feather("ihra_lexicon.feather")

In [24]:
dict(zip(ihra_lexicon_df["ihra_section"],ihra_lexicon_df["lexicon_chapter"]))

{2: [2],
 3: [2, 7],
 4: [2, 9],
 5: [2],
 6: [2],
 7: [2],
 8: [2],
 9: [6],
 11: [2],
 12: [2],
 13: [2, 9],
 14: [2],
 15: [4],
 16: [4],
 18: [4],
 21: [4],
 23: [2, 7],
 24: [10],
 25: [11],
 26: [2, 7],
 27: [0, 1],
 28: [10],
 29: [2, 7],
 30: [7],
 31: [7],
 33: [8],
 34: [7],
 35: [7],
 36: [9],
 40: [0],
 41: [0]}